In [2]:
# ============================================================
# CELL 1 - Environment Setup
# Install all required dependencies for MMSegmentation + SegFormer
# ============================================================

# Install PyTorch with CUDA 12.1 support
!pip install torch==2.4.1 torchvision==0.19.1 --index-url https://download.pytorch.org/whl/cu121 -q

# Install OpenMMLab stack
!pip install mmengine -q
!pip install mmcv==2.2.0 -f https://download.openmmlab.com/mmcv/dist/cu121/torch2.4/index.html -q
!pip install mmsegmentation -q

# Fix mmcv version check (mmseg 1.2.2 hardcodes max version as 2.2.0 exclusive)
!sed -i 's/MMCV_MAX = .2.2.0./MMCV_MAX = "2.3.0"/' \
    /usr/local/lib/python3.12/dist-packages/mmseg/__init__.py

# Install text encoding dependencies
!pip install ftfy -q

# Install additional augmentation library
!pip install albumentations -q

print("Installation complete, version check fixed!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 798.9/798.9 MB 801.1 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.1/7.1 MB 62.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 75.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 57.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 106.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 1.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 7.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 MB 13.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.2/124.2 MB 7.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.0/196.0 MB 6.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 176.2/176.2 MB 6.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━

In [3]:
# ============================================================
# CELL 2 - Environment Verification
# Verify all packages are correctly installed and GPU is available
# ============================================================

import torch, mmcv, mmengine, mmseg

print('torch:', torch.__version__)
print('mmcv:', mmcv.__version__)
print('mmseg:', mmseg.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

torch: 2.4.1+cu121
mmcv: 2.2.0
mmseg: 1.2.2
CUDA available: True
GPU: Tesla T4


In [4]:
# ============================================================
# CELL 3 - Mount Google Drive and Extract Dataset
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

# Extract dataset from Drive to Colab local storage (faster I/O during training)
!unzip -q /content/drive/MyDrive/Dataset/processed.zip -d /content/

# Verify dataset structure and image counts
print("=== Dataset Verification ===")
!echo "Train images:" && ls /content/processed/train/images/ | wc -l
!echo "Train masks: " && ls /content/processed/train/masks/  | wc -l
!echo "Val images:  " && ls /content/processed/val/images/   | wc -l
!echo "Val masks:   " && ls /content/processed/val/masks/    | wc -l
!echo "Test images: " && ls /content/processed/test/images/  | wc -l
!echo "Test masks:  " && ls /content/processed/test/masks/   | wc -l

Mounted at /content/drive
=== Dataset Verification ===
Train images:
672
Train masks: 
672
Val images:  
149
Val masks:   
149
Test images: 
146
Test masks:  
146


In [12]:
# ============================================================
# CELL 4 - Clone Repository and Download Pretrained Weights
# ============================================================
import os
import shutil

repo_path = '/content/repo'

# 1. Move out of the directory before deleting it to avoid "No such file or directory"
%cd /content

# 2. Clean up existing repo safely
if os.path.exists(repo_path):
    shutil.rmtree(repo_path)
    print(f"Cleaned up existing repository at {repo_path}")

# 3. Clone the project repository
!git clone https://github.com/ilMassy/advertising-panel-segmentation {repo_path}

# 4. Change directory into the new repo
%cd {repo_path}

# 5. Create necessary directories using absolute paths to be safe
os.makedirs(os.path.join(repo_path, 'results'), exist_ok=True)
os.makedirs(os.path.join(repo_path, 'models'), exist_ok=True)

# 6. Download MIT-B1 pretrained weights
!wget -q --show-progress \
    https://download.openmmlab.com/mmsegmentation/v0.5/pretrain/segformer/mit_b1_20220624-02e5a6a1.pth \
    -O checkpoints/mit_b1_imagenet.pth

print("\nSetup completed successfully!")
!ls -lh checkpoints/

/content
Cloning into '/content/repo'...
remote: Enumerating objects: 255, done.
remote: Counting objects: 100% (65/65), done.
remote: Compressing objects: 100% (45/45), done.
remote: Total 255 (delta 33), reused 40 (delta 15), pack-reused 190 (from 1)
Receiving objects: 100% (255/255), 246.89 KiB | 6.33 MiB/s, done.
Resolving deltas: 100% (131/131), done.
/content/repo
checkpoints/mit_b1_imagenet.pth: No such file or directory

Setup completed successfully!
ls: cannot access 'checkpoints/': No such file or directory


In [7]:
# ============================================================
# CELL 5 - Restore B1 Results from Drive
# Needed for comparison in evaluation cell
# ============================================================

import os

# Create results directory
os.makedirs('/content/repo/results/exp1_segformer_b1_standard', exist_ok=True)

# Copy best B1 checkpoint from Drive to Colab local storage
!cp /content/drive/MyDrive/advertising_panel_segmentation/results/exp1_segformer_b1_standard/best_mIoU_iter_10000.pth \
    /content/repo/results/exp1_segformer_b1_standard/

# Verify
print("B1 checkpoint restored:")
!ls -lh /content/repo/results/exp1_segformer_b1_standard/

B1 checkpoint restored:
total 54M
-rw------- 1 root root 54M Mar 25 08:10 best_mIoU_iter_10000.pth


In [13]:
# ============================================================
# CELL 6 - Verify Repository Structure
# ============================================================

import os

# Verify all files were cloned correctly
!echo "=== Config files ===" && ls /content/repo/configs/
!echo "=== Source scripts ===" && ls /content/repo/src/
!echo "=== Results directory ===" && ls /content/repo/results/
!echo "=== Models directory ===" && ls /content/repo/models/

# Verify dataset
!echo "=== Dataset ==="
!ls /content/processed/

print("Repository structure verified!")

=== Config files ===
segformer_b0_baseline.py  segformer_b1_augmented.py  segformer_b1_standard.py
=== Source scripts ===
check_masks.py	CVAT_preparation.py  extract_frames.py	reorder_by_prefix.py
=== Results directory ===
exp0_segformer_b0_baseline  exp1_segformer_b1_standard
=== Models directory ===
README.md
=== Dataset ===
test  train  val
Repository structure verified!


In [ ]:
# ============================================================
# CELL 7 - Create Augmented Config
# SegFormer-B1 with sport-specific augmentations (Phase 4)
# Motion Blur + Occlusion Cutout + Color Jitter (HueSaturationValue
# + RandomBrightnessContrast + PhotoMetricDistortion)
# ============================================================

import os
os.makedirs('/content/repo/configs', exist_ok=True)

segformer_b1_augmented_config = """
_base_ = [
    'mmseg::_base_/models/segformer_mit-b0.py',
    'mmseg::_base_/default_runtime.py',
]

# Dataset settings
dataset_type = 'BaseSegDataset'
data_root    = '/content/processed/'
crop_size    = (640, 640)

# Training pipeline with sport-specific augmentations
train_pipeline = [
    dict(type='LoadImageFromFile'),
    dict(type='LoadAnnotations'),
    dict(type='RandomResize', scale=(1920, 1080), ratio_range=(0.5, 2.0), keep_ratio=True),
    dict(type='RandomCrop', crop_size=crop_size, cat_max_ratio=0.75),
    dict(type='RandomFlip', prob=0.5),
    dict(type='PhotoMetricDistortion'),
    # Sport-specific augmentations via albumentations
    dict(
        type='Albu',
        transforms=[
            # Motion Blur: simulates camera panning during match
            dict(type='MotionBlur', blur_limit=9, p=0.4),
            # Gaussian Noise: simulates broadcast video compression
            dict(type='GaussNoise', var_limit=(10.0, 50.0), p=0.3),
            # Color Jitter: simulates stadium lighting variations
            dict(type='RandomBrightnessContrast',
                 brightness_limit=0.3,
                 contrast_limit=0.3, p=0.4),
            # Color Jitter: simulates different cameras and artificial lights
            dict(type='HueSaturationValue',
                 hue_shift_limit=20,
                 sat_shift_limit=30,
                 val_shift_limit=20, p=0.3),
            # Occlusion Cutout: simulates players occluding the board
            dict(type='CoarseDropout',
                 max_holes=6,
                 max_height=80,
                 max_width=80,
                 min_holes=1,
                 fill_value=0, p=0.3),
        ],
        keymap={'img': 'image', 'gt_semantic_seg': 'mask'},
        update_pad_shape=False,
    ),
    dict(type='PackSegInputs'),
]

# Validation/Test pipeline (no augmentation)
val_pipeline = [
    dict(type='LoadImageFromFile'),
    dict(type='Resize', scale=(1920, 1080), keep_ratio=True),
    dict(type='LoadAnnotations'),
    dict(type='PackSegInputs'),
]

# Dataset meta info (2 classes: background and board)
meta = dict(
    classes=('background', 'board'),
    palette=[[0, 0, 0], [255, 0, 0]]
)

# Dataloaders
train_dataloader = dict(
    batch_size=4, num_workers=2,
    dataset=dict(
        type=dataset_type, data_root=data_root,
        data_prefix=dict(img_path='train/images', seg_map_path='train/masks'),
        img_suffix='.jpeg', seg_map_suffix='.png',
        metainfo=meta, pipeline=train_pipeline,
    )
)

val_dataloader = dict(
    batch_size=1, num_workers=2,
    dataset=dict(
        type=dataset_type, data_root=data_root,
        data_prefix=dict(img_path='val/images', seg_map_path='val/masks'),
        img_suffix='.jpeg', seg_map_suffix='.png',
        metainfo=meta, pipeline=val_pipeline,
    )
)

# Test set sepeared from validation
test_dataloader = dict(
    batch_size=1, num_workers=2,
    dataset=dict(
        type=dataset_type, data_root=data_root,
        data_prefix=dict(img_path='test/images', seg_map_path='test/masks'),
        img_suffix='.jpeg', seg_map_suffix='.png',
        metainfo=meta, pipeline=val_pipeline,
    )
)

# Evaluation metrics
val_evaluator  = dict(type='IoUMetric', iou_metrics=['mIoU', 'mDice'])
test_evaluator = dict(type='IoUMetric', iou_metrics=['mIoU', 'mDice', 'mFscore'])

# Model: SegFormer-B1 architecture
model = dict(
    data_preprocessor=dict(
        type='SegDataPreProcessor',
        mean=[123.675, 116.28, 103.53],
        std=[58.395, 57.12, 57.375],
        bgr_to_rgb=True,
        pad_val=0,
        seg_pad_val=255,
        size=crop_size,
        size_divisor=None,
    ),
    backbone=dict(
        embed_dims=64,
        num_heads=[1, 2, 5, 8],
        num_layers=[2, 2, 2, 2],
        init_cfg=dict(
            type='Pretrained',
            checkpoint='/content/repo/checkpoints/mit_b1_imagenet.pth'
        )
    ),
    decode_head=dict(
        in_channels=[64, 128, 320, 512],
        num_classes=2,
    ),
    test_cfg=dict(mode='whole')
)

# AdamW optimizer
optimizer = dict(type='AdamW', lr=6e-5, betas=(0.9, 0.999), weight_decay=0.01)
optim_wrapper = dict(
    type='OptimWrapper',
    optimizer=optimizer,
    paramwise_cfg=dict(
        custom_keys={
            'pos_block': dict(decay_mult=0.),
            'norm':      dict(decay_mult=0.),
            'head':      dict(lr_mult=10.)
        }
    )
)

# LR scheduler with warmup
param_scheduler = [
    dict(type='LinearLR', start_factor=1e-6, by_epoch=False, begin=0,    end=1500),
    dict(type='PolyLR',   eta_min=0.0,       power=1.0,      begin=1500, end=20000, by_epoch=False),
]

# Save best checkpoint based on mIoU
default_hooks = dict(
    checkpoint=dict(type='CheckpointHook', by_epoch=False, interval=2000,
                    max_keep_ckpts=3, save_best='mIoU', rule='greater'),
    logger=dict(type='LoggerHook', interval=50),
)

train_cfg = dict(type='IterBasedTrainLoop', max_iters=20000, val_interval=2000)
val_cfg   = dict(type='ValLoop')
test_cfg  = dict(type='TestLoop')
"""

# Write config to disk
with open('/content/repo/configs/segformer_b1_augmented.py', 'w') as f:
    f.write(segformer_b1_augmented_config)

print("Config created!")
print("\nAugmentations included:")
print("Motion Blur       → camera panning during match")
print("Gaussian Noise    → broadcast video compression")
print("Color Jitter      → RandomBrightnessContrast + HueSaturationValue + PhotoMetricDistortion")
print("Occlusion Cutout  → players occluding the board")
print("\nDataset splits:")
print("train → train/images + train/masks")
print("val   → val/images   + val/masks   (used during training)")
print("test  → test/images  + test/masks  (used for final evaluation)")
!ls -lh /content/repo/configs/

Config created!

Augmentations included:
Motion Blur       → camera panning during match
Gaussian Noise    → broadcast video compression
Color Jitter      → RandomBrightnessContrast + HueSaturationValue + PhotoMetricDistortion
Occlusion Cutout  → players occluding the board

Dataset splits:
train → train/images + train/masks
val   → val/images   + val/masks   (used during training)
test  → test/images  + test/masks  (used for final evaluation)
total 16K
-rw-r--r-- 1 root root 3.3K Mar 24 19:06 segformer_b0_baseline.py
-rw-r--r-- 1 root root 5.0K Mar 24 20:40 segformer_b1_augmented.py
-rw-r--r-- 1 root root 3.5K Mar 24 19:06 segformer_b1_standard.py


In [ ]:
# ============================================================
# CELL 8 - Push Augmented Config to GitHub
# ============================================================

import os
from getpass import getpass

# Git configuration
!cd /content/repo && git config user.email "massimilianogiangreco7@gmail.com"
!cd /content/repo && git config user.name "ilMassy"

# Ask for token securely (input is hidden)
GITHUB_TOKEN = getpass("Enter your GitHub Personal Access Token: ")

# Set remote URL with authentication token
!cd /content/repo && git remote set-url origin \
    https://{GITHUB_TOKEN}@github.com/ilMassy/advertising-panel-segmentation.git

# Pull remote changes first
!cd /content/repo && git pull origin main --rebase

# Add and commit config file
!cd /content/repo && git add configs/
!cd /content/repo && git commit -m "Add SegFormer-B1 augmented config (Phase 4) - Motion Blur, Occlusion Cutout, Color Jitter" \
    || echo "Nothing to commit, already done"

# Push to GitHub
!cd /content/repo && git push origin main

print("Config file pushed to GitHub!")

Enter your GitHub Personal Access Token: ··········
From https://github.com/ilMassy/advertising-panel-segmentation
 * branch            main       -> FETCH_HEAD
Already up to date.
On branch main
Your branch is up to date with 'origin/main'.

nothing to commit, working tree clean
Nothing to commit, already done
Everything up-to-date
Config file pushed to GitHub!


In [ ]:
# ============================================================
# CELL 9 - Training: SegFormer-B1 Augmented (Phase 4)
# Fine-tuning B1 with sport-specific augmentations
# Checkpoints saved directly to Google Drive during training
# ============================================================

import os
import mmseg

# Correct path for mmseg tools on Colab
mmseg_tools = os.path.join(
    os.path.dirname(mmseg.__file__),
    '.mim', 'tools', 'train.py'
)
print(f"MMSeg train tool: {mmseg_tools}")

# Verify tool exists
assert os.path.exists(mmseg_tools), f"train.py not found at {mmseg_tools}"
print("Train tool found!")

# Create output directory on Drive
!mkdir -p /content/drive/MyDrive/advertising_panel_segmentation/results/exp2_segformer_b1_augmented

# Launch B1 augmented training
# work-dir points directly to Drive so checkpoints are saved during training
!python {mmseg_tools} \
    /content/repo/configs/segformer_b1_augmented.py \
    --work-dir /content/drive/MyDrive/advertising_panel_segmentation/results/exp2_segformer_b1_augmented \
    2>&1 | tee /content/training_augmented_log.txt

MMSeg train tool: /usr/local/lib/python3.12/dist-packages/mmseg/.mim/tools/train.py
Train tool found!
/usr/local/lib/python3.12/dist-packages/mmengine/optim/optimizer/zero_optimizer.py:11: DeprecationWarning: `TorchScript` support for functional optimizers is deprecated and will be removed in a future PyTorch release. Consider using the `torch.compile` optimizer instead.
  from torch.distributed.optim import \
03/24 20:41:25 - mmengine - INFO - 
------------------------------------------------------------
System environment:
    sys.platform: linux
    Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
    CUDA available: True
    MUSA available: False
    numpy_random_seed: 393508950
    GPU 0: Tesla T4
    CUDA_HOME: /usr/local/cuda
    NVCC: Cuda compilation tools, release 12.8, V12.8.93
    GCC: x86_64-linux-gnu-gcc (Ubuntu 11.4.0-1ubuntu1~22.04.3) 11.4.0
    PyTorch: 2.4.1+cu121
    PyTorch compiling details: PyTorch built with:
  - GCC 9.3
  - C++ Version: 201703
  - Inte

In [ ]:
# ============================================================
# CELL 9 - Verify Augmented Results on Google Drive
# ============================================================

import os

drive_results = "/content/drive/MyDrive/advertising_panel_segmentation/results/exp2_segformer_b1_augmented"

print("Checkpoints saved on Drive:")
!ls -lh {drive_results}/*.pth 2>/dev/null || echo "  No checkpoints yet"

print("\nLast checkpoint:")
!cat {drive_results}/last_checkpoint 2>/dev/null || echo "  Not found"

Checkpoints saved on Drive:
-rw------- 1 root root  55M Mar 24 23:16 /content/drive/MyDrive/advertising_panel_segmentation/results/exp2_segformer_b1_augmented/best_mIoU_iter_14000.pth
-rw------- 1 root root 159M Mar 24 23:37 /content/drive/MyDrive/advertising_panel_segmentation/results/exp2_segformer_b1_augmented/iter_16000.pth
-rw------- 1 root root 160M Mar 24 23:59 /content/drive/MyDrive/advertising_panel_segmentation/results/exp2_segformer_b1_augmented/iter_18000.pth
-rw------- 1 root root 160M Mar 25 00:22 /content/drive/MyDrive/advertising_panel_segmentation/results/exp2_segformer_b1_augmented/iter_20000.pth

Last checkpoint:
/content/drive/MyDrive/advertising_panel_segmentation/results/exp2_segformer_b1_augmented/iter_20000.pth

In [ ]:
# ============================================================
# CELL 10 - Evaluate B1 Augmented with Full Metrics
# mIoU, mDice, Precision, Recall on TEST SET
# Uses best checkpoint saved on Google Drive
# ============================================================

import mmseg, os, glob

mmseg_test = os.path.join(
    os.path.dirname(mmseg.__file__),
    '.mim', 'tools', 'test.py'
)

# Automatically find best checkpoint on Drive
checkpoints = glob.glob(
    '/content/drive/MyDrive/advertising_panel_segmentation/results/exp2_segformer_b1_augmented/best_mIoU_iter_*.pth'
)

if not checkpoints:
    print("No checkpoint found! Check Drive path.")
else:
    best_checkpoint = max(
        checkpoints,
        key=lambda x: int(x.split('iter_')[1].split('.')[0])
    )
    print(f"Using checkpoint: {best_checkpoint}")

    !python {mmseg_test} \
        /content/repo/configs/segformer_b1_augmented.py \
        {best_checkpoint}

Using checkpoint: /content/drive/MyDrive/advertising_panel_segmentation/results/exp2_segformer_b1_augmented/best_mIoU_iter_14000.pth
/usr/local/lib/python3.12/dist-packages/mmengine/optim/optimizer/zero_optimizer.py:11: DeprecationWarning: `TorchScript` support for functional optimizers is deprecated and will be removed in a future PyTorch release. Consider using the `torch.compile` optimizer instead.
  from torch.distributed.optim import \
03/25 00:25:10 - mmengine - INFO - 
------------------------------------------------------------
System environment:
    sys.platform: linux
    Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
    CUDA available: True
    MUSA available: False
    numpy_random_seed: 1900141130
    GPU 0: Tesla T4
    CUDA_HOME: /usr/local/cuda
    NVCC: Cuda compilation tools, release 12.8, V12.8.93
    GCC: x86_64-linux-gnu-gcc (Ubuntu 11.4.0-1ubuntu1~22.04.3) 11.4.0
    PyTorch: 2.4.1+cu121
    PyTorch compiling details: PyTorch built with:
  - GCC 9.3


In [14]:
# ============================================================
# CELL 11 - Save Logs and Model Summary to Repository
# Copies B1 augmented training logs from Drive to GitHub repo
# ============================================================

import os, glob, shutil

# ── Create directories ────────────────────────────────────────────────────────
os.makedirs('/content/repo/results/exp2_segformer_b1_augmented', exist_ok=True)
os.makedirs('/content/repo/models', exist_ok=True)

# ── Copy training logs from Drive to results/ ─────────────────────────────────
src = '/content/drive/MyDrive/advertising_panel_segmentation/results/exp2_segformer_b1_augmented'
dst = '/content/repo/results/exp2_segformer_b1_augmented'

log_files = glob.glob(f'{src}/**/*.log', recursive=True)
if log_files:
    for log_file in log_files:
        shutil.copy(log_file, dst)
        print(f"Copied: {os.path.basename(log_file)} → results/exp2_segformer_b1_augmented/")
else:
    print(f"No log files found in {src}")

# ── Update model summary in models/ ──────────────────────────────────────────
# Read existing README to preserve B0 and B1 standard entries
existing = ""
readme_path = '/content/repo/models/README.md'
if os.path.exists(readme_path):
    with open(readme_path, 'r') as f:
        existing = f.read()

# Append B1 augmented entry
b1_aug_entry = """
## Exp2 - SegFormer-B1 Augmented
- Best checkpoint: best_mIoU_iter_14000.pth
- mIoU: 87.26%
- board IoU: 76.45%
- Dice: 86.66%
- Precision: 85.76%
- Recall: 87.57%
- Stored on: Google Drive
"""

if 'Exp2' not in existing:
    with open(readme_path, 'a') as f:
        f.write(b1_aug_entry)
    print("Updated models/README.md with Exp2 entry")
else:
    print("Exp2 entry already present in models/README.md")

# ── Verify no .pth files exist ───────────────────────────────────────────────
pth_files = glob.glob('/content/repo/results/**/*.pth', recursive=True)
if pth_files:
    print("WARNING: .pth files found! Removing...")
    for f in pth_files:
        os.remove(f)
        print(f"Removed: {f}")
else:
    print("No .pth files found - safe to push!")

# ── Push to GitHub ────────────────────────────────────────────────────────────
from getpass import getpass
GITHUB_TOKEN = getpass("Enter GitHub token: ")

!cd /content/repo && git config user.email "massimilianogiangreco7@gmail.com"
!cd /content/repo && git config user.name "ilMassy"
!cd /content/repo && git remote set-url origin \
    https://{GITHUB_TOKEN}@github.com/ilMassy/advertising-panel-segmentation.git
!cd /content/repo && git pull origin main --rebase

!cd /content/repo && git add results/exp2_segformer_b1_augmented/*.log
!cd /content/repo && git add models/README.md

print("\n=== Files to be committed ===")
!cd /content/repo && git diff --cached --name-only

!cd /content/repo && git commit -m "Add B1 augmented training logs and model summary (Phase 4)" \
    || echo "Nothing to commit"
!cd /content/repo && git push origin main
print("Pushed to GitHub!")

Copied: 20260324_204124.log → results/exp2_segformer_b1_augmented/
Updated models/README.md with Exp2 entry
No .pth files found - safe to push!
Enter GitHub token: ··········
error: cannot pull with rebase: You have unstaged changes.
error: please commit or stash them.

=== Files to be committed ===
models/README.md
results/exp2_segformer_b1_augmented/20260324_204124.log
[main e22510b] Add B1 augmented training logs and model summary (Phase 4)
 2 files changed, 1843 insertions(+)
 create mode 100644 results/exp2_segformer_b1_augmented/20260324_204124.log
Enumerating objects: 11, done.
Counting objects: 100% (11/11), done.
Delta compression using up to 2 threads
Compressing objects: 100% (5/5), done.
Writing objects: 100% (7/7), 22.83 KiB | 5.71 MiB/s, done.
Total 7 (delta 3), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (3/3), completed with 3 local objects.
To https://github.com/ilMassy/advertising-panel-segmentation.git
   7969d9e..e22510b  main -> main
Pushed to 